<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">02 Data Communication</h1>
<hr>


## **7. Path Loss of Radio Communication**

<div class="alert alert-light">

  **Name:** (add name here)
  
  **Matriculation Number:** (add matriculation number here)
  
</div>


### 7.0 Prerequisites

This notebook builds on logarithms, power ratios, and the concepts of noise and signal-to-noise ratio. Run the setup cell below first, then work through the examples in order. The received-power graph uses the functions defined in Section 7.4.4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

### 7.1 Introduction

A radio signal becomes weaker as it travels, which limits the distance over which a receiver can recover a message reliably. Estimating this reduction is essential when choosing transmit power, antennas, and the spacing between devices. **Path loss** describes the attenuation associated with propagation between the transmitting and receiving antennas.

This notebook derives **free-space path loss (FSPL)** as a baseline for an unobstructed link. It then expresses the result in decibels and combines transmit power, antenna gains, and losses in a link budget. The examples connect these calculations to receiver sensitivity, nominal communication range, and the fading reserve needed for a practical design.


### 7.2 Propagation of Wireless Signals

An ideal isotropic antenna radiates equally in all directions. At distance $d$, its power is spread over a sphere with surface area $4\pi d^2$. In lossless free space, the total power crossing this sphere remains constant, but the power per unit area decreases as the sphere grows. This geometric spreading explains why a distant receiving antenna captures less power. Real antennas need not radiate equally in every direction; their directional gain is introduced in Section 7.4.1.

Radio communication uses electromagnetic (EM) waves, which can propagate without a material medium. In a linearly polarized plane wave in free space, the electric field $\mathbf{E}$ and magnetic field $\mathbf{B}$ oscillate in perpendicular directions. Both can be modelled to be perpendicular to the direction of propagation, represented by the wave vector $\mathbf{k}$. The propagation speed is approximately $c = 3\times10^8\ \mathrm{m/s}$.

<center>
<img src="../../.admin/notebooks/picture/07_EM-Wave.gif" 
     alt="EM Wave" 
     width="45%" 
     style="filter: grayscale(100%);">
<img src="../../.admin/notebooks/picture/07_Inverse_square_law.svg" 
     alt="Free Space Path Loss" 
     width="45%" 
     style="filter: grayscale(100%);">
</center>


#### 7.2.1 Free-Space Path Loss (FSPL) Equation

The following derivation assumes ideal isotropic antennas, unobstructed propagation, and sufficient separation for the far-field approximation. It excludes reflections, absorption, and additional hardware losses. The first step is to express the spherical area over which the transmitted power is distributed:
$$
A_{\text{sphere}} = 4\pi d^2
$$

Dividing the transmit power $P_t$ by this area gives the power density $S(d)$:
$$
S(d) = \frac{P_t}{A_{\text{sphere}}} = \frac{P_t}{4\pi d^2}
$$
A receiving antenna captures power over its effective aperture $A_{\text{eff}}$. The received power is
$$
P_r = S(d)\cdot A_{\text{eff}},
$$
where $A_{\text{eff}}$ is the antenna’s effective aperture. For an ideal isotropic antenna,
$$
A_{\text{eff}} = \frac{\lambda^2}{4\pi}.
$$

Substituting the power density and effective aperture gives
$$
P_r = \frac{P_t}{4\pi d^2}\cdot \frac{\lambda^2}{4\pi}
= P_t\left(\frac{\lambda}{4\pi d}\right)^2.
$$

The free-space path loss is the ratio of transmitted to received power:
$$
\text{FSPL} = \frac{P_t}{P_r} = \left(\frac{4\pi d}{\lambda}\right)^2.
$$

Using $\lambda = \dfrac{c}{f}$ (with $c$ the speed of light and $f$ the frequency):
$$
\text{FSPL} = \left(\frac{4\pi d f}{c}\right)^2.
$$

For the same isotropic antenna gains, the linear loss ratio grows with the square of distance and frequency. Doubling either quantity multiplies this ratio by four. The frequency dependence includes the receiving antenna's effective aperture, which decreases with wavelength squared for fixed gain; it does not mean that empty space absorbs more energy at higher frequencies.


### 7.3 Decibels and Power Levels

As power spreads over large distances, received powers become very small. It is convenient to use a logarithmic scale: decibels (dB).

> What is a decibel?  
> The "Bel" honors Alexander Graham Bell and is defined as $\log_{10}$ of a ratio.  
> A decibel (dB) is one-tenth of a Bel:
> $$
> \text{dB} = 10\log_{10}\!\left(\frac{P}{P_{\text{ref}}}\right).
> $$
> dB is unitless (it represents a ratio).

Converting:
$$
x_{\text{dB}} = 10\log_{10}(x), \qquad x = 10^{x_{\text{dB}}/10}.
$$

A value in **dB** expresses a power ratio, such as a gain or loss. A value in **dBm** expresses a power level relative to 1 mW, while **dBW** uses 1 W as its reference. Thus, 1 mW is 0 dBm, and 1 W is 30 dBm or 0 dBW. A negative dBm value indicates power below 1 mW, not negative physical power.

| Change in power level | Approximate power ratio |
|---|---|
| +3 dB | Twice the power |
| +6 dB | Four times the power |
| +10 dB | Ten times the power |
| −3 dB | Half the power |

An increase in path loss has the opposite effect on received power: 3 dB more loss means approximately half as much received power when the transmitter and antennas remain unchanged.

#### 7.3.1 FSPL in dB
$$
\text{FSPL(dB)} = 20\log_{10}(d) + 20\log_{10}(f) + 20\log_{10}\!\left(\frac{4\pi}{c}\right).
$$


Use $d$ in metres, $f$ in Hertz, and $c$ in metres per second in this expression. Changing units without adjusting the constant gives an incorrect result.

The distance and frequency contributions add independently in dB. Multiplying distance by a factor $a$ adds $20\log_{10}(a)$ dB of loss; multiplying frequency by a factor $b$ adds $20\log_{10}(b)$ dB. Consequently, curves at different fixed frequencies have a constant vertical separation in dB. Their distance dependence is logarithmic, so a plot against linear distance is curved, while a plot against logarithmic distance is straight.


### 7.4 Link Budget

A link budget is a simple accounting of all the gains and losses from the transmitter to the receiver. It predicts the received power and helps check if a radio link meets the required receiver sensitivity for the target data rate, with some safety margin.

**Receiver sensitivity** is the minimum received signal power needed to meet a specified reception-quality target, such as a maximum bit or packet error rate. It is usually stated in dBm for a particular modulation, coding scheme, data rate, and set of test conditions. A signal may still be detected below this level, but the required decoding performance is no longer assured.

A more negative sensitivity value means that the receiver can operate with a weaker signal. For example, a sensitivity of $-90\ \mathrm{dBm}$ is better than $-80\ \mathrm{dBm}$: it permits reception at one tenth of the signal power under the specified conditions. Sensitivity depends on the receiver's noise level and the signal-to-noise ratio (SNR) required for decoding, so it can change with bandwidth and communication mode.

In the received-power graph below, the horizontal dashed line marks the selected sensitivity. Where a received-power curve lies above that line, the simplified model meets the power requirement. The intersection marks the nominal range limit. Moving the sensitivity slider changes this assumed requirement and the corresponding range; it does not change the transmitted or received signal power. An actual receiver's sensitivity is a property of its hardware and operating mode, rather than an arbitrary adjustable threshold. The fading margin introduced below provides a reserve above the sensitivity requirement.

#### 7.4.1 Antenna Gain

An antenna converts electrical signals into radiated electromagnetic waves and performs the reverse operation during reception. Its **gain** describes performance in a particular direction relative to an ideal isotropic antenna. Gain combines radiation efficiency and directionality: concentrating radiation in one direction can increase received power along that direction without creating additional transmit power.

The linear gain factors $G_t$ and $G_r$ describe the transmit and receive antennas in the directions of the link. A gain of 1 corresponds to 0 dBi, where **dBi** means decibels relative to isotropic. Linear gain factors multiply in the link budget; gains expressed in dBi are added to the power level in dBm. Antenna orientation therefore matters when applying a gain value to a real link.

#### 7.4.2 Link Budget Equation

Including antenna gains in the free-space result gives the Friis transmission equation in linear form:
$$
P_r \;=\; P_t \, G_t \, G_r \left(\frac{\lambda}{4\pi d}\right)^2.
$$

Using the path-loss term $\text{FSPL}=\left(\frac{4\pi d}{\lambda}\right)^2$, this becomes
$$
P_r \;=\; \frac{P_t \, G_t \, G_r}{\text{FSPL}}.
$$

It is often written in decibels for easy addition/subtraction:
$$
P_r(\text{dBm}) \;=\; P_t(\text{dBm}) \;+\; G_t(\text{dBi}) \;+\; G_r(\text{dBi})
\;-\; \text{FSPL(dB)} \;-\; L_{\text{misc}}(\text{dB}),
$$
where $L_{\text{misc}}$ groups everything else that needs to be subtracted (cable/connector loss, polarization mismatch, wall/body loss, implementation loss, etc.).


For isotropic antennas and no additional losses, the expression reduces to $P_r=P_t-\text{FSPL}$ when powers are in dBm and loss is in dB. The maximum allowable path loss is therefore the difference between transmit power and receiver sensitivity. Solving the free-space equation for distance gives the nominal range used by the interactive graph:

$$
d_{\max}=\frac{c}{4\pi f}\,10^{(P_t-P_{\mathrm{sensitivity}})/20},
$$

where $P_t$ and $P_{\mathrm{sensitivity}}$ are numerical power levels in dBm, $f$ is in Hertz, and $d_{\max}$ is in metres. Increasing transmit power by $\Delta P$ dB multiplies the nominal range by $10^{\Delta P/20}$ if all other parameters remain fixed. At fixed transmit power and sensitivity, nominal range is inversely proportional to frequency.

This range is the point where received power just meets sensitivity. The model does not include terrain, atmospheric absorption, antenna misalignment, or fading, so the plotted distances are theoretical comparisons rather than coverage guarantees.

#### 7.4.3 Fading Margin and Reliable Reception

The received power predicted by a link budget is a nominal value. In a real channel, it can vary as the transmitter, receiver, or surrounding objects move. Reflected paths can combine destructively, and obstacles can temporarily weaken the signal. These variations can cause reception to fail even when the nominal power exceeds the receiver's sensitivity.

The **available link margin** is the difference between the predicted received power and the minimum power required by the receiver for the selected modulation and data rate:

$$
M_{\mathrm{available}}\,[\mathrm{dB}]
= P_{r,\mathrm{nominal}}\,[\mathrm{dBm}]
- P_{\mathrm{sensitivity}}\,[\mathrm{dBm}].
$$

For example, a predicted received power of $-65\ \mathrm{dBm}$ and a sensitivity of $-80\ \mathrm{dBm}$ give an available margin of $15\ \mathrm{dB}$. The signal can fall by 15 dB before reaching the sensitivity threshold, provided the receiver's noise and interference conditions remain unchanged. A link with **0 dB of margin** is already at that threshold: even a small additional loss can make reception unreliable. The maximum distance calculated by equating received power with sensitivity is therefore a nominal limit, not a dependable operating range.

A **fading margin**, also called a **fade margin**, is the reserve required to tolerate a chosen amount of signal fading. If the design requires a fade margin $M_f$, the link budget should satisfy

$$
P_{r,\mathrm{nominal}}\,[\mathrm{dBm}]
\geq P_{\mathrm{sensitivity}}\,[\mathrm{dBm}] + M_f\,[\mathrm{dB}].
$$

Equivalently, $M_{\mathrm{available}}\geq M_f$. In the example above, a required fade margin of 10 dB leaves another 5 dB of headroom. Reserving this margin does not change the physical received power; it makes the design requirement stricter. Known losses, such as cable attenuation, belong in $L_{\mathrm{misc}}$ and should not be counted again as an additional fading allowance.

A reasonable margin depends on the propagation environment, movement, and the acceptable probability of an outage. For the exercise, **20 dB is a defensible illustrative design target** if accompanied by assumptions about the expected fading and required reliability. It is not a universal guarantee. For context, [Campbell Scientific's guide to link budgets and fade margin](https://s.campbellsci.com/documents/us/technical-papers/link-budget.pdf) recommends a 20–30 dB design goal for highly reliable RF telemetry links and advises improving links with less than 10 dB of margin. Measurements or a suitable propagation model are needed to justify a target for a particular deployment.

More margin can be obtained by reducing the distance, improving antenna gain or placement, reducing known losses, or selecting a more robust communication mode with a lower required receive power. Increasing transmit power also helps where the hardware and permitted operating limits allow it. A practical design therefore uses the link budget to meet both the sensitivity threshold and the required fading reserve.


#### 7.4.4 Example Tool: Calculating and Plotting Free-Space Path Loss

The functions below implement the free-space path-loss equation in linear form and in decibels. Supply distance `d` in metres and frequency `f` in Hertz. `fspl_linear` returns the dimensionless ratio of transmitted to received power for isotropic antennas; `fspl_db` expresses that loss in dB. Both functions accept a single value or NumPy arrays, so the same tool can evaluate one link or generate a complete path-loss curve.

Run the cell to define the functions, calculate an example at 100 m and 2.4 GHz, and plot path loss at 2.4 GHz, 5 GHz, and 60 GHz over distances from 1 to 1000 m. The horizontal axis uses a linear distance scale, while the vertical axis shows path loss in dB. Compare the shapes of the curves and their vertical separation, using the logarithmic FSPL equation in Section 7.3.1 to explain these features.

All three curves assume ideal isotropic antennas and unobstructed free-space propagation. Antenna gains, other losses, and the required fading margin must still be considered in the link budget. The later received-power plot reuses these functions.


In [ ]:
c = 3e8  # speed of light [m/s]

def fspl_linear(d, f, c=c):
    """Free-space power-loss ratio for distance in metres and frequency in Hz."""
    d = np.asarray(d, dtype=float)
    f = np.asarray(f, dtype=float)
    return (4 * np.pi * d * f / c) ** 2

def fspl_db(d, f, c=c):
    """Free-space path loss in dB for the same distance and frequency inputs."""
    return 10 * np.log10(fspl_linear(d, f, c=c))


def checked_fspl_db(distances, frequency):
    """Check that the path-loss calculation returns a finite value for each distance."""
    try:
        values = np.asarray(fspl_db(distances, frequency), dtype=float)
    except (TypeError, ValueError):
        print("Check that the distances and frequency are numerical values.")
        return None
    if values.shape != np.shape(distances):
        print("Check that fspl_db() returns one value per distance.")
        return None
    if not np.all(np.isfinite(values)):
        print("Check fspl_db(): all path-loss values must be finite numbers.")
        return None
    return values


example_distance = 100.0  # metres
example_frequency = 2.4e9  # Hz
print(f"At {example_distance:.0f} m and {example_frequency / 1e9:.1f} GHz:")
print(f"  Linear path-loss ratio: {fspl_linear(example_distance, example_frequency):.3e}")
print(f"  Path loss: {fspl_db(example_distance, example_frequency):.2f} dB")


# Distances: 1 m to 1000 m (linearly spaced)
d = np.linspace(1, 1000, 300)

# Frequencies of interest (Hz)

f_list = [
    (2.4e9, "2.4 GHz"),
    (5e9, "5 GHz"),
    (60e9, "60 GHz"),
]

def plot_path_loss(frequencies):
    if not frequencies:
        print("Add at least one (frequency, label) entry to f_list, then rerun this cell.")
        return

    curves = []
    for frequency, label in frequencies:
        values = checked_fspl_db(d, frequency)
        if values is None:
            return
        curves.append((label, values))

    plt.figure(figsize=(7, 4.8))
    for label, values in curves:
        plt.plot(d, values, label=label)
    plt.xlabel("Distance d [m]")
    plt.ylabel("FSPL [dB]")
    plt.title("Free-Space Path Loss vs Distance (linear distance axis)")
    plt.grid(True, ls=":")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_path_loss(f_list)


---

### 7.5 Hands-On Exercises

Use the equations and graphs to explain the results, and include units with numerical values. Treat all displayed ranges as nominal free-space limits unless a fading margin is explicitly included.

<div class="alert alert-block alert-info">Fill in the responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Explain the path-loss graph in Section 7.4.4. Why do the 2.4 GHz, 5 GHz, and 60 GHz curves have the same shape and a constant vertical separation in dB? Why are they curved when distance is plotted on a linear axis? Relate the observations to the FSPL equation.**

(answer here)


Run the following cell to compare **received power** at 900 MHz, 2.4 GHz, 5 GHz, and 60 GHz. Unlike the path-loss plot above, this graph uses a logarithmic distance axis and shows power in dBm. All four curves use the same transmit power and assume isotropic antennas with no additional losses.

The red dashed line marks receiver sensitivity. Each coloured dot marks a curve's intersection with that line, and the legend reports the corresponding nominal range. Use the power and sensitivity sliders to examine how the curves and intersections change. The graph does not reserve a fading margin.


In [ ]:
from ipywidgets import FloatSlider, HBox
from IPython.display import display

def make_link_budget_figure(tx_power_dbm=30, sensitivity_dbm=-80):
    frequencies = [
        (900e6, "900 MHz", "tab:blue"),
        (2.4e9, "2.4 GHz", "tab:orange"),
        (5e9, "5 GHz", "tab:green"),
        (60e9, "60 GHz", "tab:purple"),
    ]
    available_path_loss = tx_power_dbm - sensitivity_dbm
    ranges = [
        c / (4 * np.pi * frequency) * 10**(available_path_loss / 20)
        for frequency, _, _ in frequencies
    ]
    distances = np.logspace(np.log10(min(1.0, min(ranges) / 2)), 6, 600)

    fig, ax = plt.subplots(figsize=(9, 5.2))
    for (frequency, label, color), max_distance in zip(frequencies, ranges):
        path_loss = checked_fspl_db(distances, frequency)
        if path_loss is None:
            plt.close(fig)
            return
        received_power = tx_power_dbm - path_loss
        ax.semilogx(
            distances, received_power, color=color,
            label=f"{label}: nominal range {max_distance:.1f} m",
        )
        ax.scatter(max_distance, sensitivity_dbm, color=color, s=35, zorder=3)

    ax.axhline(
        sensitivity_dbm, color="red", linestyle="--",
        label="Receiver sensitivity",
    )
    ax.set_xlabel("Distance [m]")
    ax.set_ylabel("Received power [dBm]")
    ax.set_title("Free-Space Received Power at Four Frequencies")
    ax.grid(True, which="both", linestyle=":")
    ax.legend(fontsize=9)
    fig.tight_layout()
    return fig

power_slider = FloatSlider(
    min=0, max=40, step=1, value=30,
    description="Power [dBm]", continuous_update=False,
)
sensitivity_slider = FloatSlider(
    min=-110, max=-50, step=1, value=-80,
    description="Sensitivity", continuous_update=False,
)
display(HBox([power_slider, sensitivity_slider]))
print("Dots mark the nominal ranges at 0 dB link margin; no fading reserve is included.")

link_budget_figure = make_link_budget_figure(
    power_slider.value, sensitivity_slider.value,
)
# Close the pyplot figure before explicit display to avoid automatic duplicate output.
plt.close(link_budget_figure)
link_budget_handle = display(link_budget_figure, display_id=True)

def update_link_budget(change=None):
    figure = make_link_budget_figure(power_slider.value, sensitivity_slider.value)
    if figure is None:
        return
    plt.close(figure)
    link_budget_handle.update(figure)

power_slider.observe(update_link_budget, names="value")
sensitivity_slider.observe(update_link_budget, names="value")


**What is sensitivity?**

(answer here)

**A drone transmits at 2.4 GHz with a power of 30 dBm to a receiver with a sensitivity of $-80\ \mathrm{dBm}$. Assume isotropic antennas, free-space propagation, no additional losses, and no fading margin. Set the graph's sliders to these values and use the 2.4 GHz curve for the following two questions. At 500 m, does the predicted received power meet the receiver's sensitivity requirement? State the approximate received power and the maximum nominal range of this link.**

(answer here)

**Increase transmit power from 30 dBm to 36 dBm, keeping the other parameters fixed. What is the new maximum nominal range, and by what factor has it increased? Explain the change using the free-space path-loss equation.**

(answer here)


**OPTIONAL: Have all tasks in this notebook been read and understood? Are there any shortcomings, missing parts, or sections that should be improved?**

Please help the teaching team improve the notebook by identifying unclear tasks or explanations and suggesting improvements.

(answer here)
